In [ ]:
import pandas as pd
import numpy as np
import os
from itertools import combinations

In [ ]:
def get_link_list(dir):
    linklist = []
    for file in os.listdir(dir):
        link = file.split(' ')[4]
        linklist.append(link)
    return linklist

In [ ]:
#comparar se os datasets tem o mesmo valor, fora os valores imputados 
caminho_pasta_original = '../datasets/choosen-best-svd'
caminho_pasta_imputado = '../datasets/imputed-choosen-best-svd'

tecnicas = ['knn', 'svd', 'interpolacao-linear', 'media-movel', 'mediana-movel']

lista_tcp = ['bbr']  
# lista_link = ['ap-ma', 'ba-es', 'es-ba', 'rj-ap', 'es-ba', 'es-ce', 'rj-es', 'rj-sc']  

lista_link = get_link_list(caminho_pasta_original)


def encontrar_arquivo(pasta, substrings):
    arquivos_encontrados = []
    for arquivo in os.listdir(pasta):
        if arquivo.endswith('.csv') and all(substring in arquivo for substring in substrings):
            arquivos_encontrados.append(os.path.join(pasta, arquivo))
    return arquivos_encontrados


def comparar_vazao(caminho_pasta_original, caminho_pasta_imputado, lista_tcp, lista_link, tecnicas):
    for tcp in lista_tcp:
        for tecnica in tecnicas:
            cmainho_pasta_imputado_tecnica = caminho_pasta_imputado + '/' + tecnica
            for link in lista_link:
                arquivo_original = encontrar_arquivo(caminho_pasta_original, [tcp, link])
                arquivo_imputado = encontrar_arquivo(cmainho_pasta_imputado_tecnica, [tcp, link])
                
                if arquivo_original and arquivo_imputado:
                    arquivo_original = arquivo_original[0]
                    arquivo_imputado = arquivo_imputado[0]
                    
                    df_original = pd.read_csv(arquivo_original)
                    df_original['Throughput'] = df_original['Throughput'].replace(-1, np.nan)  
                    df_imputado = pd.read_csv(arquivo_imputado)

                    if 'Throughput' not in df_original.columns or 'Throughput' not in df_imputado.columns:
                        print(f"Coluna 'throughput' não encontrada nos arquivos para tcp={tcp} e link={link}.")
                        continue
                    mask_nao_nulo = ~df_original['Throughput'].isna()
                    throughput_original = df_original.loc[mask_nao_nulo, 'Throughput']
                    throughput_imputado = df_imputado.loc[mask_nao_nulo, 'Throughput']

                    throughput_igual = throughput_original.equals(throughput_imputado)
                    
                    if throughput_igual:
                        print(f"Os dados da coluna 'throughput' para tcp={tcp} e link={link} são idênticos nas linhas não nulas.")
                    else:
                        diferencas = throughput_original != throughput_imputado
                        df_diferencas = pd.DataFrame({
                            'index': throughput_original.index[diferencas],
                            'throughput_original': throughput_original[diferencas],
                            'throughput_imputado': throughput_imputado[diferencas]
                        })
                        print(f"Diferenças encontradas para tcp={tcp} e link={link}:")
                        print(df_diferencas)
                else:
                    print(f"Arquivos para tcp={tcp} e link={link} não encontrados.")

comparar_vazao(caminho_pasta_original, caminho_pasta_imputado, lista_tcp, lista_link, tecnicas)


In [ ]:
#verificando se as imputações de diferentes metodos estao iguais
# caminho_pasta_original = '../datasets/melhores-tratados'
# caminho_pasta_imputado_base = '../datasets/dados-vazao-imputados' 

# lista_tcp = ['bbr', 'cubic']  
# lista_link = ['ap-ma', 'ba-es', 'es-ba', 'rj-ap', 'es-ba', 'es-ce', 'rj-es', 'rj-sc']  

def encontrar_arquivo(pasta, substrings):
    arquivos_encontrados = []
    for arquivo in os.listdir(pasta):
        if arquivo.endswith('.csv') and all(substring in arquivo for substring in substrings):
            arquivos_encontrados.append(os.path.join(pasta, arquivo))
    return arquivos_encontrados

def comparar_imputacao_par_a_par(caminho_pasta_original, caminho_pasta_imputado_base, lista_tcp, lista_link):
    for tcp in lista_tcp:
        for link in lista_link:
            arquivo_original = encontrar_arquivo(caminho_pasta_original, [tcp, link])
            
            if arquivo_original:
                arquivo_original = arquivo_original[0]
                
                df_original = pd.read_csv(arquivo_original)
                df_original['Throughput'] = df_original['Throughput'].replace(-1, np.nan)  

                indices_nan = df_original['Throughput'].isna()
                
                imputacoes = {}
                
                for metodo in os.listdir(caminho_pasta_imputado_base):
                    caminho_metodo = os.path.join(caminho_pasta_imputado_base, metodo)
                    if os.path.isdir(caminho_metodo): 
                        arquivo_imputado = encontrar_arquivo(caminho_metodo, [tcp, link])
                        
                        if arquivo_imputado:
                            arquivo_imputado = arquivo_imputado[0]
                            
                            df_imputado = pd.read_csv(arquivo_imputado)
                            imputacoes[metodo] = df_imputado['Throughput'][indices_nan].reset_index(drop=True)
                
                for metodo1, metodo2 in combinations(imputacoes.keys(), 2):
                    df_comparacao = pd.DataFrame({
                        metodo1: imputacoes[metodo1],
                        metodo2: imputacoes[metodo2]
                    })
                    iguais = df_comparacao[metodo1].equals(df_comparacao[metodo2])
                    if iguais:
                        print(f"Valores imputados são iguais entre {metodo1} e {metodo2} para tcp={tcp} e link={link}.")
                        print(df_comparacao)
                    else:
                        print(f"Valores diferentes entre {metodo1} e {metodo2} para tcp={tcp} e link={link}.")
            else:
                print(f"Arquivo original para tcp={tcp} e link={link} não encontrado.")

comparar_imputacao_par_a_par(caminho_pasta_original, caminho_pasta_imputado, lista_tcp, lista_link)
